# NanoGPT (Learn)

In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [2]:
import os
import sys
from pathlib import Path

from pathlib import Path

CWD = os.path.realpath(os.getcwd())
PARENT_DIR = os.path.dirname(CWD)
sys.path.append(PARENT_DIR)

DATA_DIR = Path(PARENT_DIR).parent / 'data'

In [3]:
from reader.loader import TextDataset

dataset = TextDataset(DATA_DIR, device=device)

100%|██████████| 5/5 [00:00<00:00, 825.07it/s]


In [4]:
from src.modules.architecture.ngram_lm import NgramLanguageModel
from reader.preprocess import decode
import torch

BATCH_SIZE = 16
EMBEDDING_SIZE = 64
SEQ_LENGTH = 32
DROPOUT_RATE = 0.2

N_HEADS = 4
N_BLOCKS = 4
LR = 1e-3

EVAL_ITER = 100
EVAL_INTERVAL = 100
EPOCH_SIZE = 10000

torch.manual_seed(1337)

model = NgramLanguageModel(vocab_size=dataset.vocab_size, n_heads=N_HEADS, embedding_size=EMBEDDING_SIZE, seq_length=SEQ_LENGTH, 
                            n_blocks=N_BLOCKS, dropout_rate=DROPOUT_RATE, device=device, attention_type='flash')
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

print("Number of parameters:")
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

Number of parameters:
0.212825 M parameters


In [5]:
@torch.no_grad()
def estimate_loss(x, y, model):
    losses = torch.zeros(EVAL_ITER)
    for k in range(EVAL_ITER):
        logits, loss = model(x, y)
        losses[k] = loss.item()
    return losses.mean()

In [6]:
for iter in range(EPOCH_SIZE):

    # every once in a while evaluate the loss on train and val sets
    if iter % EVAL_INTERVAL == 0 or iter == EPOCH_SIZE - 1:
        model.eval()
        x_train, y_train = dataset.load_train(batch_size=BATCH_SIZE, context_window_size=SEQ_LENGTH)
        x_val, y_val = dataset.load_test(batch_size=BATCH_SIZE, context_window_size=SEQ_LENGTH)
        train_losses = estimate_loss(x_train, y_train, model)
        val_losses = estimate_loss(x_val, y_val, model)
        print(f"step {iter}: train loss {train_losses:.4f}, val loss {val_losses:.4f}")
        model.train()

    # sample a batch of data
    x_train, y_train = dataset.load_train(BATCH_SIZE)

    # evaluate the loss
    logits, loss = model(x_train, y_train)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.6745, val loss 4.6668


KeyboardInterrupt: 

In [ ]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))

In [ ]:
from reader.preprocess import encode
context = torch.tensor([encode(dataset.stoi, 'fyodor')], dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))

In [ ]:
context = torch.tensor([encode(dataset.stoi, 'slab')], dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))

In [ ]:
torch.randint(
            100 - 10,
            (16,)
        )

In [ ]:
context = torch.tensor([encode(dataset.stoi, 'slab')], dtype=torch.long, device=device)
idx_cond = context[:, -32:]

In [ ]:
idx_cond